In [1]:
"""
Retail Demand Forecasting with External Regressors
=====================================================
Dataset: Kaggle "Walmart Recruiting - Store Sales Forecasting"
Download: https://www.kaggle.com/competitions/walmart-recruiting-store-sales-forecasting/data
Files needed (place in data/):
  - train.csv     (Store, Dept, Date, Weekly_Sales, IsHoliday)
  - features.csv  (Store, Date, Temperature, Fuel_Price, MarkDown1-5, CPI, Unemployment, IsHoliday)
  - stores.csv    (Store, Type, Size)

This gives real external regressors: Temperature (weather), IsHoliday
(holidays), MarkDown1-5 (promotions) — a genuine step up from plain
sales-history-only forecasting.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
import shap
import joblib

RANDOM_STATE = 42

# -----------------------------------------------------------------
# 1. LOAD & MERGE
# -----------------------------------------------------------------
train = pd.read_csv("data/train.csv", parse_dates=["Date"])
features = pd.read_csv("data/features.csv", parse_dates=["Date"])
stores = pd.read_csv("data/stores.csv")

# WHY: features.csv has its own IsHoliday column too (duplicate) -> drop
# one to avoid a messy _x/_y merge collision.
features = features.drop(columns=["IsHoliday"])

df = train.merge(features, on=["Store", "Date"], how="left")
df = df.merge(stores, on="Store", how="left")
df = df.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)

print(df.shape)
print(df.isna().mean().sort_values(ascending=False).head(10))


(421570, 16)
MarkDown2      0.736110
MarkDown4      0.679847
MarkDown3      0.674808
MarkDown1      0.642572
MarkDown5      0.640790
Store          0.000000
Date           0.000000
Dept           0.000000
Fuel_Price     0.000000
Temperature    0.000000
dtype: float64


In [2]:
# -----------------------------------------------------------------
# 2. EDA
# -----------------------------------------------------------------
# WHY MarkDowns are mostly NaN: promotions (MarkDown1-5) only started
# being recorded partway through the dataset's history -> NaN means
# "no promotion running", not "missing data" -> fill with 0, not mean.
markdown_cols = ["MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]
df[markdown_cols] = df[markdown_cols].fillna(0)
df["CPI"] = df["CPI"].fillna(df["CPI"].median())
df["Unemployment"] = df["Unemployment"].fillna(df["Unemployment"].median())

# Overall sales trend
weekly_total = df.groupby("Date")["Weekly_Sales"].sum()
plt.figure(figsize=(10, 4))
weekly_total.plot()
plt.title("Total weekly sales across all stores")
plt.savefig("outputs/total_sales_trend.png", bbox_inches="tight")
plt.close()

# Holiday effect
holiday_avg = df.groupby("IsHoliday")["Weekly_Sales"].mean()
print("\nAverage sales, holiday vs non-holiday week:\n", holiday_avg)

# Temperature vs sales (sampled for speed)
plt.figure(figsize=(6, 4))
sns.scatterplot(data=df.sample(5000, random_state=1), x="Temperature", y="Weekly_Sales", alpha=0.3)
plt.title("Temperature vs weekly sales (sample)")
plt.savefig("outputs/temperature_vs_sales.png", bbox_inches="tight")
plt.close()


Average sales, holiday vs non-holiday week:
 IsHoliday
False    15901.445069
True     17035.823187
Name: Weekly_Sales, dtype: float64


In [3]:
# -----------------------------------------------------------------
# 3. FEATURE ENGINEERING
# -----------------------------------------------------------------
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)
df["Quarter"] = df["Date"].dt.quarter

# WHY lag features: last week's and last year's same-week sales are
# usually the strongest predictors in retail forecasting — capture both
# short-term momentum and yearly seasonality (e.g. holiday spikes).
df = df.sort_values(["Store", "Dept", "Date"])
grp = df.groupby(["Store", "Dept"])["Weekly_Sales"]
df["lag_1"] = grp.shift(1)
df["lag_52"] = grp.shift(52)  # same week, previous year
df["rolling_mean_4"] = grp.transform(lambda s: s.shift(1).rolling(4).mean())

df["MarkDown_total"] = df[markdown_cols].sum(axis=1)
df["IsHoliday"] = df["IsHoliday"].astype(int)
df["Type"] = df["Type"].astype("category").cat.codes  # A/B/C -> 0/1/2

# WHY dropna here: early rows for each Store/Dept have no lag_52 history
# (first year of data) -> can't be used for a model relying on lag
# features. This does shrink the dataset, and is worth mentioning as a
# deliberate tradeoff in your README.
df = df.dropna(subset=["lag_1", "lag_52", "rolling_mean_4"])

feature_cols = [
    "Store", "Dept", "IsHoliday", "Temperature", "Fuel_Price",
    "MarkDown_total", "CPI", "Unemployment", "Size", "Type",
    "Year", "Month", "WeekOfYear", "Quarter",
    "lag_1", "lag_52", "rolling_mean_4",
]
X = df[feature_cols]
y = df["Weekly_Sales"]


In [4]:
# -----------------------------------------------------------------
# 4. CHRONOLOGICAL TRAIN/TEST SPLIT
# -----------------------------------------------------------------
# WHY NOT a random split: this is time-series data. A random split lets
# the model "see the future" (train on later weeks, test on earlier
# ones), which inflates performance in a way that won't hold up in
# production. Always split by date for time series.
cutoff_date = df["Date"].quantile(0.85, interpolation="nearest")
train_mask = df["Date"] <= cutoff_date
X_train, X_test = X[train_mask], X[~train_mask]
y_train, y_test = y[train_mask], y[~train_mask]
print(f"\nTrain: {X_train.shape}, Test: {X_test.shape}, cutoff: {cutoff_date.date()}")



Train: (223218, 17), Test: (37865, 17), cutoff: 2012-07-27


In [5]:
# -----------------------------------------------------------------
# 5. NAIVE BASELINE — always compute this first
# -----------------------------------------------------------------
# WHY: "last year, same week" is what a store manager would guess with
# zero modeling. Any real model needs to beat this by a meaningful
# margin, or it isn't earning its complexity.
naive_pred = df.loc[~train_mask, "lag_52"]
naive_mae = mean_absolute_error(y_test, naive_pred)
naive_rmse = np.sqrt(mean_squared_error(y_test, naive_pred))
print(f"\nNaive baseline (same week last year) -> MAE: {naive_mae:.2f}, RMSE: {naive_rmse:.2f}")



Naive baseline (same week last year) -> MAE: 1754.75, RMSE: 3745.99


In [6]:
# -----------------------------------------------------------------
# 6. MODEL 1 — Linear Regression
# -----------------------------------------------------------------
lr = LinearRegression()
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)
print(f"\nLinear Regression -> MAE: {mean_absolute_error(y_test, pred_lr):.2f}, "
      f"RMSE: {np.sqrt(mean_squared_error(y_test, pred_lr)):.2f}")


Linear Regression -> MAE: 1486.82, RMSE: 2954.49


In [7]:

# -----------------------------------------------------------------
# 7. MODEL 2 — Random Forest
# -----------------------------------------------------------------
rf = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
print(f"Random Forest -> MAE: {mean_absolute_error(y_test, pred_rf):.2f}, "
      f"RMSE: {np.sqrt(mean_squared_error(y_test, pred_rf)):.2f}")


Random Forest -> MAE: 1236.17, RMSE: 2630.08


In [8]:
# -----------------------------------------------------------------
# 8. MODEL 3 — XGBoost
# -----------------------------------------------------------------
xgb_model = xgb.XGBRegressor(
    n_estimators=400, max_depth=6, learning_rate=0.05, random_state=RANDOM_STATE
)
xgb_model.fit(X_train, y_train)
pred_xgb = xgb_model.predict(X_test)
print(f"XGBoost -> MAE: {mean_absolute_error(y_test, pred_xgb):.2f}, "
      f"RMSE: {np.sqrt(mean_squared_error(y_test, pred_xgb)):.2f}")

XGBoost -> MAE: 1263.76, RMSE: 2675.23


In [9]:
# -----------------------------------------------------------------
# 9. WEIGHTED MAE — the domain-specific metric (Kaggle's own metric)
# -----------------------------------------------------------------
# WHY: holiday weeks matter 5x more to the business (biggest sales,
# biggest stakeout risk) — using this instead of plain MAE mirrors how
# the original Kaggle competition actually scored submissions, and
# shows you understand that not all errors cost the business equally.
weights_test = np.where(df.loc[~train_mask, "IsHoliday"] == 1, 5, 1)
def weighted_mae(y_true, y_pred, weights):
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

print(f"\nWeighted MAE (holiday weeks x5) — XGBoost: "
      f"{weighted_mae(y_test.values, pred_xgb, weights_test):.2f}")


Weighted MAE (holiday weeks x5) — XGBoost: 1312.52


In [10]:
# -----------------------------------------------------------------
# 10. FEATURE IMPORTANCE / SHAP
# -----------------------------------------------------------------
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test.sample(1000, random_state=1))
shap.summary_plot(shap_values, X_test.sample(1000, random_state=1), show=False)
plt.savefig("outputs/shap_summary.png", bbox_inches="tight")
plt.close()

In [11]:
# -----------------------------------------------------------------
# 11. SAVE ARTIFACTS FOR THE APP
# -----------------------------------------------------------------
joblib.dump(xgb_model, "outputs/xgb_forecast_model.pkl")
joblib.dump(feature_cols, "outputs/feature_columns.pkl")
# WHY not include Store/Dept separately here: feature_cols already
# contains "Store" and "Dept", so adding them again created duplicate
# columns, which broke history["Store"].unique() downstream in the app.
history_cols = ["Date", "Weekly_Sales"] + feature_cols
joblib.dump(df[history_cols].tail(20000), "outputs/recent_history.pkl")

print("\nDone. Model + artifacts saved to outputs/")



Done. Model + artifacts saved to outputs/
